# レポート課題3：Phase 1　前処理

本ノートブックは2026年度課題3に用いるコーパスを前処理し、pickleファイルを生成する Phase 1 実行用のコードである。

このノートブック内で実行する内容は、課題レポートとして報告する必要はありません。

## 全体の流れ

| フェーズ | Level | 内容 | 所要時間 | 実行頻度 |
|---------|-------|------|---------|----------|
| **Phase 1：前処理フェーズ** | 0〜1 | CSVの読み込み・spacy 処理・pickle 保存 | 数分 | 最初の1回のみ |
| **Phase 2：モデル構築・評価フェーズ** | 2〜5 | pickle 読み込み・分割・ベクトル化・学習・評価・分析 | 数秒〜数十秒 | 繰り返し可 |

このノートブックは Phase 1 のみを実行する。

### 実行手順

1. 予め Google Drive 直下に `2026dm-rep3` という名前のフォルダを作成しておく。これまで利用していた「Colab Notebooksフォルダ」の中ではないことに注意。
2. 教員から配布された `report3_job_reviews.csv` を Google Drive の `2026dm-rep3` フォルダに配置する。
3. 本ノートブックも同じフォルダ内に用意し、一回実行する。（pickle ファイルが保存される）
4. 以降の課題は pickle ファイルを読み込むだけで前処理済みの結果を利用することができる。

---

# Phase 1：前処理フェーズ

## 最初に1回だけ実行すること（所要時間：数分）

spacy（ja_ginza）で全テキストを形態素解析し、結果を Google Drive に保存する。
pickle ファイルが保存されたことを確認してから Phase 2 に進むこと。

## Level 0-0：データ取得と確認

Google Drive 直下に `2026dm-rep3` フォルダを作成し、
配布された `report3_job_reviews.csv` をそのフォルダに配置してから実行すること。

`drive.mount()` は、Googleドライブ内のファイルを実行環境から参照するための関数。ノートブック自体はGoogleドライブ内に置いて実行しているはずだが、実際の実行環境はクラウド環境のどこかに用意されており、そのままではGoogleドライブ内を参照することができない。参照場所を指定して、読み書き権限を付与することでクラウド環境からGoogleドライブ内のファイルを参照することができるようになる。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 自身の Google Drive 直下に「2026dm-rep3」フォルダを作成してから実行すること
path = "/content/drive/MyDrive/2026dm-rep3/"
print(f"保存先: {path}")
!ls {path}

In [ ]:
# GiNZA（日本語 spacy モデル）のインストール
#!pip install spacy ginza ja_ginza
!pip install -U ginza ja_ginza

In [ ]:
# GiNZAの準備
import spacy

# Python 3.12 + Spacy 3.8 + Ginza 5.2 の構成だとそのままでは動作しないため、
# 以下の設定を追加指定
config = {
    "components": {
        "compound_splitter": {
            "split_mode": "A"
        }
    }
}

nlp = spacy.load("ja_ginza", config=config)
print("GiNZA 読み込み成功:", nlp.meta["name"], nlp.meta["version"])

In [ ]:
import pandas as pd

csv_file = path + "report3_job_reviews.csv"
df_raw = pd.read_csv(csv_file, encoding="utf-8-sig")

# 感情ラベルが付与されていないサンプルを除外する
df = df_raw[df_raw["sentiment_label"].notna() & (df_raw["sentiment_label"] != "")].reset_index(drop=True)

print(f"全サンプル数: {len(df_raw)}  うちラベルあり: {len(df)} 件")
print()
print("sentiment_label の分布:")
print(df["sentiment_label"].value_counts())
print()
print("industry_type の分布:")
print(df["industry_type"].value_counts())
print()
print("review_category の分布:")
print(df["review_category"].value_counts())
print()

# df観察
df.head()

## Level 0-1：spacy による分かち書きとファイル保存

spacy（ja_ginza）で各テキストを **原形（lemma）** に変換し、スペース区切りの文字列として保存する。

### なぜ原形（lemma）を使うのか？

CountVectorizer は文字列の一致でカウントするため、
「働く」「働いた」「働ける」を別の語として扱う。
原形に統一することで活用形の違いを吸収して同じ語として集計できる。

```
入力: 「残業が多くなく、定時退社しやすい雰囲気だ」
出力: 「残業 が 多い なく 、 定時 退社 する やすい 雰囲気 だ」
```

### なぜ pickle に保存するのか？

spacy による処理は全サンプル数に比例して時間がかかる。これはサンプル数が多くなればなるほどより顕著になる。毎回同じ前処理をするのは無駄だ。処理結果をファイルに保存しておくことで、Phase 2 以降ではファイルを読み込むだけでよくなり、試行錯誤を素早く繰り返せる。

今回は1分弱のためそこまで気にならないレベルだが、同じ処理を何度も繰り返さない練習を兼ねて「途中結果を保存」する流れを導入している。

In [ ]:
print(f"分かち書き処理中... 全 {len(df)} 件")

wakati = []
for i, text in enumerate(df["review_text"]):
    tokens = [token.lemma_ for token in nlp(str(text))]
    wakati.append(" ".join(tokens))
    if (i + 1) % 200 == 0:
        print(f"  {i + 1} / {len(df)} 件処理済み")

df["wakati"] = wakati
print("\n処理完了")
display(df[["review_text", "wakati"]].head(3))

In [ ]:
import pickle

pkl_file = path + "job_reviews_add.pkl"
with open(pkl_file, "wb") as f:
    pickle.dump(df, f)

print(f"保存完了: {pkl_file}")
!ls -lh {path}

---

# Phase 2：モデル構築・評価フェーズ（コード例）

一度Phase 1を実行して pickle ファイルを用意したら、pickle.load関数を利用してファイルから読み込むことで保存しておいた前処理データを再利用することができる。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import pandas as pd

path = "/content/drive/MyDrive/2026dm-rep3/"
pkl_file = path + "job_reviews_add.pkl"

with open(pkl_file, "rb") as f:
    df = pickle.load(f)

print(f"読み込み完了: {len(df)} 件")
df.head()